In [2]:
import numpy as np
import networkx as nx
from scipy.spatial.distance import cosine

np.random.seed(42)

## Step 1: Generate Vector Data

Create normalized vectors that represent documents or data points in a high-dimensional space.

In [3]:
# Create a small dataset

N = 500
D = 32

data = np.random.randn(N, D)

# Normalize the data
lengths = np.linalg.norm(data, axis=1, keepdims=True)
data = data / lengths

# HNSW settings
M = 8
ef_search = 20
p = 0.5

print("Number of vectors:", N)
print("Vector dimension:", D)

Number of vectors: 500
Vector dimension: 32


## Step 2: Build Multi-Layer HNSW Index

Assign vectors to different layers using probability-based layer selection and connect each vector to its nearest neighbors.

In [5]:
def build_hnsw(data, M=8, p=0.5):

    N = len(data)

    # Assign random levels using probability p

    levels = []

    for _ in range(N):

        level = 0

        while np.random.rand() < p:
            level += 1

        levels.append(level)

    max_level = max(levels)

    # Create graph for each layer

    graphs = [
        nx.Graph()
        for _ in range(max_level + 1)
    ]

    # Add nodes to their assigned layers

    for i, level in enumerate(levels):

        for l in range(level + 1):
            graphs[l].add_node(i)

    # Connect each node to nearest neighbors

    for l, G in enumerate(graphs):

        nodes = list(G.nodes())

        for i in nodes:

            similarities = data[nodes] @ data[i]

            order = np.argsort(
                similarities
            )[::-1]

            neighbors = [
                nodes[j]
                for j in order
                if nodes[j] != i
            ][:M]

            for j in neighbors:
                G.add_edge(i, j)

    entry = int(np.argmax(levels))

    return graphs, levels, entry


graphs, levels, entry = build_hnsw(
    data,
    M=M,
    p=p
)

print("HNSW index created.")
print("Number of layers:", len(graphs))
print("Entry point:", entry)

HNSW index created.
Number of layers: 11
Entry point: 288


## Step 3: Search the HNSW Index

Traverse the hierarchical graph from the highest layer to the lowest layer and return the vectors with the highest cosine similarity.

In [6]:
def hnsw_search(
    query,
    data,
    graphs,
    entry,
    ef_search=20,
    top_k=5
):

    query = query / np.linalg.norm(query)

    current = entry

    # Search from highest layer to lowest layer

    for level in range(
        len(graphs) - 1,
        -1,
        -1
    ):

        G = graphs[level]

        improved = True

        while improved:

            improved = False

            neighbors = list(
                G.neighbors(current)
            )

            if not neighbors:
                break

            scores = data[neighbors] @ query

            best = neighbors[
                np.argmax(scores)
            ]

            if data[best] @ query > data[current] @ query:
                current = best
                improved = True

    # Final search at layer 0

    G = graphs[0]

    visited = {current}
    candidates = [current]
    scores = {
        current: float(data[current] @ query)
    }

    while candidates and len(visited) < ef_search:

        node = candidates.pop()

        for neighbor in G.neighbors(node):

            if neighbor in visited:
                continue

            visited.add(neighbor)

            scores[neighbor] = float(
                data[neighbor] @ query
            )

            candidates.append(neighbor)

    results = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )[:top_k]

    return results

## Step 4: Run a Vector Search

Generate a query vector and retrieve the top matching vectors from the HNSW index.

In [7]:
# Create a random query vector

query = np.random.randn(D)

# Search the HNSW index

results = hnsw_search(
    query,
    data,
    graphs,
    entry,
    ef_search=ef_search,
    top_k=5
)

print("Top-5 Results:")
print("ID\tCosine Similarity")

for idx, score in results:

    print(
        f"{idx}\t{score:.4f}"
    )

Top-5 Results:
ID	Cosine Similarity
162	0.4674
366	0.4199
461	0.3852
31	0.3461
170	0.3447


## Conclusion

The HNSW vector indexing system was implemented from scratch using NumPy and NetworkX. The implementation creates multiple graph layers using probability-based level assignment, performs hierarchical traversal, and returns top-K vectors using cosine similarity for fast semantic retrieval.